In [0]:
FIELDS = ['bootes', 'ursaminor', 'draco']

CONFIG_FILES = ['/home/enk/pfs/ga_targeting/configs/netflow/SSP/dSph/_common.py']
for field in FIELDS:
    CONFIG_FILES.append(f'/home/enk/pfs/ga_targeting/configs/netflow/SSP/dSph/{field}.py')

# Imports and preliminaries

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pfs.ga.targeting.targets.dsph import GALAXIES as DSPH_FIELDS
from pfs.ga.targeting.instrument import *
#from pfs.ga.targeting import config
from pfs.ga.common.diagram import FOV
from pfs.ga.common.projection import WcsProjection
from astropy.time import Time
import logging

In [0]:
plt.rc('font', size=6)
cmap = plt.get_cmap('tab10')

# Cobra_charmer requires a valid logging handler
logging.getLogger().addHandler(logging.NullHandler())

obs_time = Time('2026-01-12T10:00:00')        # midnight in Hawaii; This variable needs to be defined, but it does not need to be set to any particular date.


# Load the SSP observation logs

In [0]:
from pfs.ga.targeting.targets.read_obslog_csvs import read_all_obslog_csvs

# Read all CSV files into a single dataframe
obslogs = read_all_obslog_csvs()

# Define the function that processes a single field

In [0]:
def get_ga_obslog(pnames, pointings, obslogs):
    """
    Create a DataFrame with the observation logs for each pointing.
    """
    ga_obslog = pd.DataFrame()
    ga_obslog['pointing_name'] = pnames
    ga_obslog['ra'] = [pp.ra for pp in pointings]
    ga_obslog['dec'] = [pp.dec for pp in pointings]
    ga_obslog['dates'] = [[] for pp in pointings]
    ga_obslog['tot_exptime'] = [0.0 for pp in pointings]
    ga_obslog['tot_eet_b'] = [0.0 for pp in pointings]
    ga_obslog['tot_eet_r'] = [0.0 for pp in pointings]
    ga_obslog['tot_eet_m'] = [0.0 for pp in pointings]
    ga_obslog['tot_eet_n'] = [0.0 for pp in pointings]
    ga_obslog['avg_seeing'] = [0.0 for pp in pointings]
    ga_obslog['avg_transparency'] = [0.0 for pp in pointings]
    ga_obslog['avg_exptime'] = [[] for pp in pointings]
    ga_obslog['eet_b'] = [[] for pp in pointings]
    ga_obslog['eet_r'] = [[] for pp in pointings]
    ga_obslog['eet_m'] = [[] for pp in pointings]
    ga_obslog['eet_n'] = [[] for pp in pointings]
    ga_obslog['seeing_median'] = [[] for pp in pointings]
    ga_obslog['transparency_median'] = [[] for pp in pointings]
    ga_obslog['visit_id'] = [[] for pp in pointings]
    ga_obslog['design_id'] = [[] for pp in pointings]


    for pname in ga_obslog['pointing_name']:
        galoc = ga_obslog.index[ga_obslog['pointing_name'] == pname][0]
        observation_mask = obslogs['sequence_name'].str.startswith(pname, na=False)
        observations = obslogs[observation_mask].index
        for observation in observations:
            ga_obslog.loc[galoc, 'dates'].append(obslogs.loc[observation, 'issued_at'])
            if obslogs.loc[observation, 'avg_exptime'] > 0:
                ga_obslog.loc[galoc, 'tot_exptime'] += obslogs.loc[observation, 'avg_exptime']        
            if obslogs.loc[observation, 'eet_b'] > 0:
                ga_obslog.loc[galoc, 'tot_eet_b'] += obslogs.loc[observation, 'eet_b']
            if obslogs.loc[observation, 'eet_r'] > 0:
                ga_obslog.loc[galoc, 'tot_eet_r'] += obslogs.loc[observation, 'eet_r']
            if obslogs.loc[observation, 'eet_m'] > 0:
                ga_obslog.loc[galoc, 'tot_eet_m'] += obslogs.loc[observation, 'eet_m']
            if obslogs.loc[observation, 'eet_n'] > 0:
                ga_obslog.loc[galoc, 'tot_eet_n'] += obslogs.loc[observation, 'eet_n']
            ga_obslog.loc[galoc, 'avg_exptime'].append(obslogs.loc[observation, 'avg_exptime'])
            ga_obslog.loc[galoc, 'eet_b'].append(obslogs.loc[observation, 'eet_b'])
            ga_obslog.loc[galoc, 'eet_r'].append(obslogs.loc[observation, 'eet_r'])
            ga_obslog.loc[galoc, 'eet_m'].append(obslogs.loc[observation, 'eet_m'])
            ga_obslog.loc[galoc, 'eet_n'].append(obslogs.loc[observation, 'eet_n'])
            ga_obslog.loc[galoc, 'seeing_median'].append(obslogs.loc[observation, 'seeing_median'])
            ga_obslog.loc[galoc, 'transparency_median'].append(obslogs.loc[observation, 'transparency_median'])
            ga_obslog.loc[galoc, 'visit_id'].append(obslogs.loc[observation, 'visit_id'])
            ga_obslog.loc[galoc, 'design_id'].append(obslogs.loc[observation, 'source_file'])
        seeing_masked = np.ma.masked_array(ga_obslog.loc[galoc, 'seeing_median'], mask=np.isnan(ga_obslog.loc[galoc, 'seeing_median']))
        weights_masked = np.ma.masked_array(ga_obslog.loc[galoc, 'avg_exptime'], mask=np.isnan(ga_obslog.loc[galoc, 'seeing_median']))
        ga_obslog.loc[galoc, 'avg_seeing'] = np.ma.average(seeing_masked, weights=weights_masked)
        transparency_masked = np.ma.masked_array(ga_obslog.loc[galoc, 'transparency_median'], mask=np.isnan(ga_obslog.loc[galoc, 'transparency_median']))
        weights_masked = np.ma.masked_array(ga_obslog.loc[galoc, 'avg_exptime'], mask=np.isnan(ga_obslog.loc[galoc, 'transparency_median']))
        ga_obslog.loc[galoc, 'avg_transparency'] = np.ma.average(transparency_masked, weights=weights_masked)
    return ga_obslog

In [0]:
def plot_field(field, pointings, pnames, ga_obslog):
    config = field.get_netflow_config()
    config.load(CONFIG_FILES, ignore_collisions=True)

    pointing = field.get_center()
    wcs = WcsProjection(pointing, proj='TAN')
    fov = FOV(projection=wcs)
    f, ax = plt.subplots(1, 1, figsize=(2.3, 2.3), dpi=240, subplot_kw=dict(projection=wcs.wcs))
    f.subplots_adjust(left=0.2, right=0.95, top=0.9, bottom=0.1)

    alpha = 0.2
    s = np.s_[::10]

    for pp, pname in zip(pointings, pnames):
        pp.obs_time = obs_time
        wfc = SubaruWFC(pp)
        pfi = SubaruPFI(wfc, instrument_options=config.instrument_options)

        pfi.plot_focal_plane(ax, fov, corners=True, c='k', ls='-', lw=0.75, label=f'Stage {pp.stage}')

        if ga_obslog.loc[ga_obslog['pointing_name'] == pname, 'tot_exptime'].values[0] > 0:
            pfi.plot_focal_plane(ax, fov, corners=True, c='k', ls='-', lw=0.75, label=f'Stage {pp.stage}', fill=True, fillcolor='g', fillalpha=0.3)


    # Plot the repeated pointings in red
    for pp in pointings:
        if field.name == 'Ursa Minor' and pp.stage < 2 or \
        field.name == 'Draco' and pp.stage < 1 or \
        field.name == 'Sextans' and pp.stage < 3 or \
        field.name == 'Bootes I' or \
        field.name == 'Sculptor' and pp.stage < 1 or \
        field.name == 'NGC 6822' and pp.stage < 2 or \
        field.name == 'Fornax' and pp.stage < 1:
            pass
        else:
            continue

        pp.obs_time = obs_time
        wfc = SubaruWFC(pp)
        pfi = SubaruPFI(wfc, instrument_options=config.instrument_options)

        pfi.plot_focal_plane(ax, fov, corners=True, c='r', ls='-', lw=0.75, label=f'Stage {pp.stage}')        

    ax.set_xlim(-3, 1)
    ax.set_ylim(-3, 1)

    ax.invert_xaxis()

    # Move all ticks inside
    ax.tick_params(direction="in") #, top=True, right=True)

    # ax.grid()
    # ax.legend()

    # f.tight_layout()

    ax.set_title(field.name, fontsize=6)
    # f.savefig(f'dsph_pointings_{FIELD}.pdf') #, bbox_inches="tight")

In [0]:
def run_pointing(field, obslogs):
    # Load the netflow configuration for the field

    pointings = field.get_pointings(SubaruPFI)

    # Match the pointings to the obslogs
    pnames = []
    stages = [pp.stage for pp in pointings]
    pstages = {}
    for stage in np.unique(stages):
        pstages[stage] = 0
    for pp in pointings:
        pnames.append(f"SSP_GA_{field.ID}_S{pp.stage:01d}P{pstages[pp.stage]:02d}")
        pstages[pp.stage] += 1

    ga_obslog = get_ga_obslog(pnames, pointings, obslogs)

    plot_field(field, pointings, pnames, ga_obslog)

    return ga_obslog


In [0]:
ga_obslogs = None
for FIELD in FIELDS:
    field = DSPH_FIELDS[FIELD]
    print(field.name)
    ga_obslog = run_pointing(field, obslogs)

    if ga_obslogs is not None:
        ga_obslogs = pd.concat([ga_obslogs, ga_obslog], ignore_index=True)
    else:
        ga_obslogs = ga_obslog

ga_obslogs.head()